# P06 · Comparación entre exchanges
**QuantLab | Módulo 1 — Fundamentos financieros | Práctica 6/12**

Kraken · OKX · Bitvavo | BTC/EUR spot | Nivel introductorio/intermedio

## 1. Objetivos
Investigar si los tres mercados se mueven conjuntamente, si uno parece anticipar a otro y si las diferencias de cotización sobreviven a costes de ejecución. No vamos a enviar órdenes ni utilizar claves privadas.

- Comprender la diferencia entre correlación, lead-lag y arbitraje.
- Alinear cotizaciones sin utilizar datos futuros.
- Calcular retornos, márgenes y capacidad visible al mejor precio.
- Separar descubrimiento y validación cronológica.

**Modo de lectura:** ejecuta las celdas en orden, interpreta sus resultados y revisa los ejercicios resueltos al final. El modo inicial es sintético, con un retraso de 2 segundos introducido deliberadamente en OKX; no constituye evidencia real. El modo `live` captura una muestra breve de cotizaciones públicas (no histórico). El modo `csv` permite reutilizar el recolector del laboratorio mediante un esquema explícito.

Las funciones están en **`src/quantlab/p06/cross_exchange.py`** y sus pruebas en **`tests/test_cross_exchange.py`** (`pytest` desde la raíz del proyecto).

## 2. Configuración
En Colab los paquetes suelen estar disponibles. En local instala, si es necesario, `pip install pandas numpy matplotlib requests`. Todas las fechas se interpretan en UTC. Las comisiones son **supuestos didácticos**, no tarifas vigentes. Las cotizaciones se comparan en EUR por BTC; el tamaño está en BTC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, time
from itertools import permutations
from pathlib import Path
from IPython.display import display
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p06.cross_exchange import (align_quotes, capture_report, collect_live, delay_receipt, episodes,
                                         fee_sensitivity, fetch_quote, fresh_returns, lag_profile, leadlag_summary,
                                         manual_cross_correlation, mid_and_spread, parse_quote, route_summary,
                                         scan_pair, scan_routes, size_comparison, synthetic_quotes, validate_quotes)

DATA_MODE = "synthetic"  # "live" o "csv"
CSV_PATH = "p06_quotes.csv"
EXPORT = False
EXCHANGES = ["kraken", "okx", "bitvavo"]
SYMBOL = "BTC/EUR"
GRID = "1s"
MAX_AGE = pd.Timedelta(seconds=2)
MAX_SKEW = pd.Timedelta(milliseconds=750)
LIVE_ROUNDS = 60
FEE_BPS = {e: 10.0 for e in EXCHANGES}  # por lado
EXTRA_BPS = 1.0  # reserva adversa por lado, además del spread
QTY = 0.01
MAX_LAG = 5
OKX_BASE = "https://www.okx.com"
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "grid.alpha": .22, "font.size": 11})

## 3. Datos originales y adaptadores
Una cotización L1 incluye el mejor comprador (`bid`), el mejor vendedor (`ask`) y sus cantidades visibles (`bid_size`, `ask_size`). No es el libro completo. Registramos `received_at` cuando la respuesta llega al sistema, `request_at` antes de solicitarla y `exchange_at` cuando el proveedor informa de su timestamp. No inventamos timestamps ausentes.

Documentación: [Kraken Ticker](https://support.kraken.com/articles/360000920306-api-symbols-and-tickers), [OKX API v5](https://www.okx.com/docs-v5/en/#order-book-trading-market-data-get-ticker), [Bitvavo ticker book](https://docs.bitvavo.com/docs/rest-api/get-ticker-book/). Si un dominio regional o mercado no está disponible, registra el error en vez de sustituir BTC/EUR por BTC/USDT. El dominio de OKX se puede configurar para tu región según su documentación.

El modo vivo hace solicitudes concurrentes por ronda, no simultáneas a nivel de motor. REST puede entregar datos cacheados o retrasados; para investigación a segundos se requerirán WebSockets, relojes sincronizados y control de secuencias.

> ⚠️ **Kraken:** el tamaño que devuelve el endpoint `Ticker` (`b`/`a`[2]) vale siempre `1.000`, no la cantidad real (ya lo vimos en P01). Por eso `fetch_quote` lee el mejor nivel del libro (`Depth`, `count=1`). Comprobado el 19-11-2026: el `Ticker` daba 1,000 / 1,000 BTC y el libro, 0,006 / 0,273 BTC en el mismo instante. Con el `Ticker`, el escáner aceptaría tamaños que no existen.

In [ ]:
print(inspect.getsource(parse_quote))
print(inspect.getsource(fetch_quote))

### 3.1. Mercado didáctico reproducible
Simulamos una trayectoria común con incrementos independientes. Kraken recibe la trayectoria sin retraso artificial; OKX con dos segundos y Bitvavo con uno. Añadimos ruido, tamaños variables, algunos desajustes extremos y pérdidas de observaciones. El propósito es comprobar que nuestro método reconoce un fenómeno conocido, no estimar un edge real.

In [ ]:
print(inspect.getsource(synthetic_quotes))

if DATA_MODE == "synthetic":
    raw = synthetic_quotes(seed=61, exchanges=EXCHANGES, symbol=SYMBOL)
    errors = pd.DataFrame()
elif DATA_MODE == "live":
    raw, errors = collect_live(LIVE_ROUNDS, EXCHANGES, OKX_BASE)  # los fallos quedan en `errors`
elif DATA_MODE == "csv":
    raw = pd.read_csv(CSV_PATH)
    errors = pd.DataFrame()
else: raise ValueError("DATA_MODE desconocido")
if not errors.empty: display(errors.head(10))
print("Modo:", DATA_MODE, "| Filas:", len(raw))
display(raw.head())

## 4. Calidad y normalización
El CSV debe incluir `exchange,symbol,received_at,bid,ask,bid_size,ask_size`. Opcionales: `request_at,exchange_at`. Todos los precios y tamaños deben ser positivos y finitos; no se aceptan libros cruzados ni timestamps de recepción duplicados para un exchange. No se fabrican cotizaciones para rellenar huecos. Una hora UTC sin zona en un CSV solo debe usarse si el dato original realmente está en UTC.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿las cotizaciones tienen sentido?</b>

`validate_quotes` se detiene con un error si:

- Faltan columnas, falta alguno de los tres exchanges o aparece otro par distinto de BTC/EUR.
- Hay precios o tamaños ≤ 0 o no finitos, o un libro cruzado (ask < bid).
- Falta una hora de recepción, o se repite para un mismo exchange.
- Una solicitud es posterior a su recepción (relojes inconsistentes).

Añade `mid`, `spread_bps` y `rtt_ms` (tiempo entre solicitud y recepción).

</div>

In [ ]:
print(inspect.getsource(validate_quotes))

quotes=validate_quotes(raw, EXCHANGES, SYMBOL)
display(quotes.groupby("exchange").agg(rows=("mid","size"), spread_bps=("spread_bps","median"), rtt_ms=("rtt_ms","median")))

## 5. Alineación temporal causal
Para cada segundo $t$, utilizamos únicamente la última cotización recibida antes de o en $t$ (`merge_asof`, dirección `backward`). No utilizamos `nearest`, que podría seleccionar datos futuros. Se limita la antigüedad a dos segundos; si el timestamp del exchange está disponible se descartan también los eventos demasiado antiguos o futuros con tolerancia de reloj de 250 ms. La antigüedad de recepción no garantiza que el evento real sea reciente. Para arbitraje exigiremos además una diferencia de recepción entre mercados inferior a 750 ms.

In [ ]:
print(inspect.getsource(align_quotes))

aligned=align_quotes(quotes, GRID, MAX_AGE, exchanges=EXCHANGES)
mid=aligned.xs("mid",axis=1,level=1)
coverage=mid.notna().mean().rename("coverage")
display(coverage.to_frame())
print("Segundos en rejilla:",len(mid))
ax=mid.iloc[:180].plot(title=f"BTC/EUR · primeros 180 segundos · {DATA_MODE}",ylabel="EUR/BTC",xlabel="UTC")
plt.show()

## 6. Correlación de retornos, no de niveles
Calculamos $r_t=\log(M_t/M_{t-1})$. Las filas vacías permanecen ausentes: no se comprimen los huecos. Para la correlación y el lead-lag también exigimos una recepción distinta de la anterior: descartamos precios simplemente arrastrados. Una cotización actualizada puede repetir el mismo precio legítimamente. Correlación no implica causalidad; una alta correlación de niveles podría deberse a una tendencia común.

In [ ]:
print(inspect.getsource(fresh_returns))

returns=fresh_returns(aligned)
correlation=returns.corr(min_periods=20)
display(correlation.round(3))
fig,ax=plt.subplots(figsize=(5.8,4))
im=ax.imshow(correlation,vmin=-1,vmax=1,cmap="coolwarm")
ax.set_xticks(range(3),EXCHANGES); ax.set_yticks(range(3),EXCHANGES)
for i in range(3):
    for j in range(3): ax.text(j,i,f"{correlation.iloc[i,j]:.2f}",ha="center",va="center",color="white" if abs(correlation.iloc[i,j])>.65 else "black")
ax.set_title(f"Correlación de retornos · {DATA_MODE}")
fig.colorbar(im,ax=ax,label="Pearson"); plt.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · La correlación simultánea es casi nula aunque los precios sean los mismos</b>

| | Kraken | OKX | Bitvavo |
|---|---|---|---|
| Kraken | 1 | 0,034 | −0,024 |
| OKX | 0,034 | 1 | −0,003 |
| Bitvavo | −0,024 | −0,003 | 1 |

- Los tres mercados siguen **la misma trayectoria**, pero desplazada: OKX va 2 s por detrás de Kraken y Bitvavo, 1 s.
- Como los incrementos de la trayectoria son independientes, el retorno de Kraken en el segundo t no tiene relación con el de OKX en ese mismo segundo. Ese retorno de Kraken reaparece en OKX en t + 2.
- **Lección:** una correlación simultánea baja **no** prueba que dos mercados sean independientes. Hay que mirar también los desplazamientos (lead-lag).

</div>

## 7. Lead-Lag: descubrimiento y validación
Definimos $\rho(k)=Corr(r^{Kraken}_t,r^{OKX}_{t+k})$. Un desplazamiento **positivo** sugiere que Kraken se adelanta a OKX; un negativo, lo contrario. Usamos `shift(-k)` en la segunda serie. Los desplazamientos son segundos porque la rejilla es regular.

Buscaremos el máximo de la correlación en el primer 65 % de los datos. Evaluaremos ese mismo desplazamiento en el resto, sin optimizarlo de nuevo. Las parejas no pueden cruzar la frontera temporal: se calcula cada desplazamiento dentro de cada segmento. Como los retornos usan el precio anterior, dejamos también un intervalo de separación. La curva de validación se reserva para un ejercicio posterior, no para elegir parámetros. No calculamos p-valores ingenuos: existe dependencia temporal y búsqueda múltiple.

In [ ]:
print(inspect.getsource(lag_profile))
print(inspect.getsource(leadlag_summary))

leadlag, both_profiles = leadlag_summary(returns, "kraken", ["okx", "bitvavo"], MAX_LAG, .65)
profiles = {target: dev for target, (dev, _) in both_profiles.items()}
display(leadlag.round(3))
fig,ax=plt.subplots()
for e,p in profiles.items(): ax.plot(p.index,p['corr'],marker='o',label=f"Kraken → {e}")
ax.axvline(0,color='gray',linestyle='--');ax.axhline(0,color='gray',linewidth=.6)
ax.set(xlabel="Desplazamiento de segundo mercado (s)",ylabel="Correlación Pearson",title=f"Lead-lag · desarrollo · {DATA_MODE}");ax.legend();plt.tight_layout();plt.show()

### 7.1. ¿Por qué cambia tanto la correlación entre periodos?
La simulación incluye dos desajustes extremos: OKX +220 € durante 6 segundos (dentro del desarrollo) y Bitvavo −230 € durante 4 segundos (dentro de la validación). Comparamos Pearson con Spearman, que usa rangos y no se deja arrastrar por unos pocos valores enormes.

In [ ]:
leadlag_spearman, _ = leadlag_summary(returns, "kraken", ["okx", "bitvavo"], MAX_LAG, .65, method="spearman")
display(pd.concat({"Pearson": leadlag.set_index("target"), "Spearman": leadlag_spearman.set_index("target")}, axis=1).round(3))
print("Mayor retorno absoluto por mercado (bps):", (returns.abs().max() * 1e4).round(1).to_dict())

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Lead-lag: el método recupera los retrasos, pero Pearson se deja arrastrar por los outliers</b>

| Mercado | Lag elegido | Pearson desarrollo | Pearson validación | Spearman desarrollo | Spearman validación |
|---|---|---|---|---|---|
| OKX | **+2 s** | 0,396 | 0,973 | **0,968** | **0,968** |
| Bitvavo | **+1 s** | 0,978 | 0,286 | **0,975** | **0,967** |

- **El método funciona:** encuentra exactamente los retrasos introducidos (+2 s en OKX, +1 s en Bitvavo). El signo positivo significa que Kraken se adelanta.
- **Pearson es frágil:** el desajuste de OKX (+220 € en el desarrollo) produce retornos de 37 bps, frente a un máximo de 3 bps en Kraken. Esos pocos puntos hunden el Pearson de desarrollo de OKX a 0,40. Con Bitvavo pasa lo mismo, pero en la validación (0,29).
- **Spearman es estable** (≈ 0,97 en los cuatro casos) porque trabaja con rangos: un retorno enorme solo cuenta como "el mayor".
- **Lección:** antes de interpretar un cambio de correlación entre periodos, hay que mirar si lo provocan unos pocos valores extremos. Aquí la relación no cambió: cambiaron los outliers.

</div>

## 8. Arbitraje ejecutable al mejor nivel: seis direcciones
Comprar A al ask y vender B al bid. Para una cantidad $q$, ajustamos conservadoramente los precios por una reserva adversa $s$ y descontamos las comisiones de cada exchange:

$$C=qAsk_A(1+s)(1+f_A),\quad V=qBid_B(1-s)(1-f_B),\quad PnL=V-C.$$

El margen neto en bps es $10^4 PnL/C$. Exigimos $q$ inferior o igual a ambos tamaños visibles. No estimamos profundidad L2 ni suponemos que el tamaño visible permanecerá disponible. La reserva adversa no es una predicción del slippage. El capital necesario incluye EUR en A y BTC en B; el margen sobre compra NO es rentabilidad sobre todo el capital. Tampoco contemplamos reequilibrado, costes fijos, redondeo, mínimos de orden, latencia real ni fallos de ejecución. Por tanto el resultado es un **candidato**, nunca un beneficio realizado.

In [ ]:
print(inspect.getsource(scan_pair))

routes=list(permutations(EXCHANGES,2))
scans=scan_routes(aligned, QTY, FEE_BPS, EXTRA_BPS, MAX_SKEW, EXCHANGES)
routes_table=route_summary(scans)
display(routes_table.round(3))
fig,ax=plt.subplots(figsize=(9,3.8));routes_table.best_net_bps.plot.barh(ax=ax,color="#4678a6")
ax.axvline(0,color="black",linewidth=.8);ax.set(xlabel="Máximo margen neto observado (bps)",title=f"Seis rutas · {QTY} BTC · {DATA_MODE}");plt.tight_layout();plt.show()

## 9. Duración y sensibilidad a costes
Un episodio es una secuencia de segundos consecutivos donde la misma ruta es candidata. Una pérdida de datos interrumpe el episodio; el número de segundos observados no equivale a duración continua de una oportunidad real. No sumamos los PnL de todas las filas: una misma oportunidad puede aparecer repetida y las operaciones agotarían el inventario.

In [ ]:
print(inspect.getsource(episodes))

candidate_episodes=episodes(scans)
display(candidate_episodes.head(15))
sensitivity=fee_sensitivity(aligned, QTY, [0,5,10,20,40], EXTRA_BPS, MAX_SKEW, EXCHANGES)
display(sensitivity)
ax=sensitivity.plot(x="fee_bps_per_side",y="candidate_route_seconds",marker="o",legend=False,title=f"Sensibilidad a comisiones · {DATA_MODE}",xlabel="Comisión por lado (bps)",ylabel="Segundos-ruta candidatos");plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Todos los candidatos con comisión salen de los desajustes inventados</b>

| Ruta | Segundos elegibles | Segundos candidatos | Mejor margen neto |
|---|---|---|---|
| Kraken → OKX | 1.611 | 5 | +14,3 bps |
| Kraken → Bitvavo | 1.627 | 0 | −19,7 bps |
| OKX → Kraken | 1.635 | 0 | −18,7 bps |
| OKX → Bitvavo | 1.638 | 0 | −20,3 bps |
| Bitvavo → Kraken | 1.638 | 3 | +16,2 bps |
| Bitvavo → OKX | 1.626 | 8 | +16,9 bps |

- Con 10 bps por lado hay **16 segundos-ruta candidatos** en **6 episodios**, pero todos ocurren en dos momentos: 00:03:21–00:03:26 (OKX +220 €) y 00:20:52–00:20:54 (Bitvavo −230 €). Son **2 eventos reales** vistos desde varias rutas.
- **Sensibilidad:** 45 candidatos con comisión 0 (30 episodios, casi todos por ruido de unos pocos euros), 16 con 5 y 10 bps, y **0 con 20 bps o más**. Con 5 y 10 bps sale lo mismo porque solo sobreviven los desajustes enormes, de unos 36 bps.
- El margen "sobre la compra" no es rentabilidad del capital: hace falta tener a la vez EUR en un exchange y BTC en el otro.

</div>

## 10. Comprobaciones reproducibles
Validamos los adaptadores con respuestas ficticias, la dirección de la alineación, el retraso introducido y la aritmética de costes. Estas pruebas no verifican la disponibilidad en vivo ni la calidad de un histórico particular.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: adaptadores, causalidad de la alineación, aritmética de costes y retrasos conocidos
k=parse_quote("kraken",{"error":[],"result":{"X":{"b":["99","0","0.2"],"a":["100","0","0.3"]}}})
kd=parse_quote("kraken",{"error":[],"result":{"X":{"bids":[["99","0.2",0]],"asks":[["100","0.3",0]]}}})
o=parse_quote("okx",{"code":"0","data":[{"instId":"BTC-EUR","bidPx":"101","askPx":"102","bidSz":"0.4","askSz":"0.5","ts":"1735689600000"}]})
b=parse_quote("bitvavo",{"market":"BTC-EUR","bid":"99","ask":"100","bidSize":"0.2","askSize":"0.3"})
assert k["bid_size"]==.2 and k["ask"]==100
assert kd["bid_size"]==.2 and kd["ask_size"]==.3
assert o["bid"]==101 and o["ask_size"]==.5 and pd.notna(o["exchange_at"])
assert b["bid"]==99 and b["ask_size"]==.3
for e in EXCHANGES:
    a=aligned[e]; known=a.received_at.notna()
    assert (a.loc[known,"received_at"]<=a.index[known]).all()
    assert a.loc[known,"age_ms"].le(MAX_AGE.total_seconds()*1000).all()
for d in scans.values():
    assert d.loc[~d.eligible,"net_eur"].isna().all()
    assert d.loc[d.candidate,"net_eur"].gt(0).all()
assert sensitivity.candidate_route_seconds.is_monotonic_decreasing
assert np.isclose(60150*(1-.001)-60000*(1+.001),29.85)
if DATA_MODE=="synthetic":
    assert leadlag.set_index("target").loc["okx","lag_s"]==2
    assert leadlag.set_index("target").loc["bitvavo","lag_s"]==1
print("Comprobaciones superadas.")

## 11. Ejercicios
| Ejercicio | Tarea | Criterio de comprobación |
|---|---|---|
| E1 | Calcula mid y spread bps para bid=59.990 y ask=60.010. Compara con `validate_quotes`. | Mid = 60.000; spread ≈ 3,333 bps. |
| E2 | Implementa tu propia correlación cruzada con `shift`. | En sintético encuentra +2 s en OKX. Explica el signo. |
| E3 | Compara tamaños 0,01, 0,05 y 0,1 BTC, sin cambiar precios. | El número de candidatos no puede aumentar con el tamaño. |
| E4 | Repite el escaneo con 0, 10 y 40 bps de comisión por lado. | Explica por qué no restamos el spread dos veces. |
| E5 | Introduce una demora artificial de 1 s en la recepción de un exchange, sin modificar precios. | ¿Cambia el lag estimado? ¿Por qué no prueba liderazgo real? |
| E6 | Ejecuta en modo live; registra errores, cobertura, RTT y número de pares válidos. | No concluyas si la muestra es insuficiente. Mercado inaccesible: registra el fallo. |
| E7 | Diseña un experimento con datos WebSocket y latencia de ejecución. | Especifica timestamps, hipótesis, benchmark, costes, inventario y validación. |

### E1 · Mid y spread

In [ ]:
print(inspect.getsource(mid_and_spread))

e1_mid, e1_spread = mid_and_spread(59990, 60010)
e1_check = validate_quotes(pd.DataFrame([dict(exchange=e, symbol=SYMBOL, received_at="2025-01-01T00:00:00Z",
                                              bid=59990, ask=60010, bid_size=1, ask_size=1) for e in EXCHANGES]),
                           EXCHANGES, SYMBOL)
display(e1_check[["exchange", "mid", "spread_bps"]])
assert e1_mid == 60000 and np.isclose(e1_spread, 10/3)
assert np.allclose(e1_check.mid, e1_mid) and np.allclose(e1_check.spread_bps, e1_spread)
print(f"E1 verificado: mid = {e1_mid:,.0f} EUR, spread = {e1_spread:.3f} bps (igual que validate_quotes).")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Mid y spread</b>

- **Mid** = (59.990 + 60.010) / 2 = **60.000 €**.
- **Spread** = (60.010 − 59.990) / 60.000 × 10.000 = 20 / 60.000 × 10.000 = **3,333 bps**.
- `validate_quotes` da exactamente lo mismo: usa la misma fórmula (spread relativo al mid, igual que en P01).

</div>

### E2 · Correlación cruzada propia

In [ ]:
print(inspect.getsource(manual_cross_correlation))

cut = int(len(returns) * .65)
e2 = pd.DataFrame({"OKX (manual)": manual_cross_correlation(returns.kraken.iloc[:cut], returns.okx.iloc[:cut], MAX_LAG),
                   "Bitvavo (manual)": manual_cross_correlation(returns.kraken.iloc[:cut], returns.bitvavo.iloc[:cut], MAX_LAG)})
display(e2.round(3).T)
assert np.allclose(e2["OKX (manual)"].dropna(), lag_profile(returns.kraken.iloc[:cut], returns.okx.iloc[:cut], MAX_LAG)["corr"].dropna())
print("Máximo OKX en", int(e2["OKX (manual)"].idxmax()), "s | máximo Bitvavo en", int(e2["Bitvavo (manual)"].idxmax()), "s")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · El máximo está en +2 s: Kraken se adelanta a OKX</b>

- La implementación propia (`shift` + fórmula de Pearson explícita) coincide con `lag_profile` y encuentra el máximo en **k = +2** para OKX (0,396) y **k = +1** para Bitvavo.
- **Signo:** ρ(k) = Corr(r^Kraken_t, r^OKX_{t+k}). Con `shift(-k)`, el retorno de OKX de **k segundos después** se pone al lado del de Kraken de ahora. Que el máximo esté en k = +2 significa que lo que hace Kraken en t, OKX lo hace en t + 2: **Kraken va por delante**.
- Un máximo en k negativo significaría lo contrario: OKX se movería primero.

</div>

### E3 · Tamaños de 0,01, 0,05 y 0,1 BTC

In [ ]:
print(inspect.getsource(size_comparison))

e3 = size_comparison(aligned, [0.01, 0.05, 0.1], FEE_BPS, EXTRA_BPS, MAX_SKEW, EXCHANGES)
display(e3)
assert e3.candidate_route_seconds.is_monotonic_decreasing and e3.eligible_route_seconds.is_monotonic_decreasing

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Más tamaño, menos oportunidades elegibles</b>

| Tamaño | Segundos-ruta elegibles | Candidatos | Episodios |
|---|---|---|---|
| 0,01 BTC | 9.775 | 16 | 6 |
| 0,05 BTC | 5.751 | 14 | 6 |
| 0,10 BTC | 2.134 | **4** | 3 |

- Con los mismos precios, una orden mayor solo puede **perder** elegibilidad: tiene que caber en el tamaño visible del ask de A **y** en el del bid de B.
- El margen en bps no depende del tamaño (los precios no cambian), así que un candidato solo desaparece si deja de caber. Nunca aparece uno nuevo.
- Con 0,1 BTC se pierde el 78 % de los segundos elegibles. En la realidad, además, una orden mayor recorrería niveles peores (slippage, P02).

</div>

### E4 · Comisiones de 0, 10 y 40 bps por lado

In [ ]:
e4 = sensitivity.set_index("fee_bps_per_side").loc[[0, 10, 40]]
display(e4)
e4_detail = {fee: route_summary(scan_routes(aligned, QTY, {e: fee for e in EXCHANGES}, EXTRA_BPS, MAX_SKEW, EXCHANGES)).candidate_seconds
             for fee in [0, 10, 40]}
display(pd.DataFrame(e4_detail).add_prefix("fee_"))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · El spread ya está dentro del ask y del bid</b>

| Comisión por lado | Segundos-ruta candidatos | Episodios |
|---|---|---|
| 0 bps | 45 | 30 |
| 10 bps | 16 | 6 |
| 40 bps | **0** | 0 |

- Cada ruta paga **dos** comisiones (una al comprar y otra al vender) más la reserva adversa de 1 bp por lado: con 10 bps, el umbral es de unos 22 bps; con 40 bps, de unos 82 bps.
- **No restamos el spread dos veces** porque ya está incluido en los precios: compramos al **ask** de A (por encima de su mid) y vendemos al **bid** de B (por debajo de su mid). Restar además el spread contaría ese coste dos veces y descartaría oportunidades reales. Es la misma idea que en P02: spread, slippage y comisiones se calculan por separado y una sola vez.
- Con comisión 0 aparecen 30 episodios por el ruido normal de unos pocos euros entre mercados. Basta una comisión realista para que desaparezcan.

</div>

### E5 · Retrasar artificialmente la recepción de un exchange

In [ ]:
print(inspect.getsource(delay_receipt))

e5_rows = []
for delayed_exchange in EXCHANGES:
    delayed = delay_receipt(raw, delayed_exchange, pd.Timedelta(seconds=1))
    d_aligned = align_quotes(validate_quotes(delayed, EXCHANGES, SYMBOL), GRID, MAX_AGE, exchanges=EXCHANGES)
    d_summary, _ = leadlag_summary(fresh_returns(d_aligned), "kraken", ["okx", "bitvavo"], MAX_LAG, .65)
    lags = d_summary.set_index("target").lag_s
    e5_rows.append({"exchange retrasado": delayed_exchange, "lag OKX": lags["okx"], "lag Bitvavo": lags["bitvavo"],
                    "cobertura del retrasado": d_aligned[delayed_exchange].valid.mean()})
display(pd.DataFrame([{"exchange retrasado": "ninguno", "lag OKX": 2, "lag Bitvavo": 1, "cobertura del retrasado": 1.0}] + e5_rows).round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · El lag mide cuándo recibimos los datos, no quién lidera el mercado</b>

| Recepción retrasada 1 s | Lag OKX | Lag Bitvavo |
|---|---|---|
| Ninguna | +2 s | +1 s |
| Bitvavo | +2 s | **+2 s** |
| OKX | **+3 s** | +1 s |
| Kraken | **+1 s** | **0 s** |

- **Sí cambia:** el lag estimado se desplaza exactamente 1 s, sin tocar ni un solo precio.
- Si retrasamos a Kraken, el "líder", Bitvavo pasa a lag 0 y OKX a +1: parece que Kraken lidera menos, pero los precios son idénticos.
- **Por eso no prueba liderazgo real:** el lead-lag se calcula con `received_at`, es decir, cuándo nos llegó el dato a nosotros. Una red más lenta, un servidor más lejano o una API con caché crean "liderazgo" sin que ningún mercado se mueva antes que otro. Para hablar de liderazgo hacen falta los timestamps del evento en cada exchange, con relojes sincronizados.
- La cobertura baja un poco (99 %): como `exchange_at` se conserva, los eventos llegan 1 s más viejos y el filtro de antigüedad descarta algunos.

</div>

### E6 · Captura real (modo live)
Cambia `LIVE_E6` a `True` para capturar `LIVE_ROUNDS` rondas (unos 60 segundos) de los tres exchanges. Los fallos de cada solicitud se registran; no se sustituyen por datos ficticios.

In [ ]:
print(inspect.getsource(capture_report))

LIVE_E6 = False

if LIVE_E6:
    live_raw, live_errors = collect_live(LIVE_ROUNDS, EXCHANGES, OKX_BASE)
    if not live_errors.empty: display(live_errors)
    live_quotes = validate_quotes(live_raw, EXCHANGES, SYMBOL)
    print("Captura UTC:", live_quotes.received_at.min(), "—", live_quotes.received_at.max())
    live_aligned = align_quotes(live_quotes, GRID, MAX_AGE, exchanges=EXCHANGES)
    live_returns = fresh_returns(live_aligned)
    live_scans = scan_routes(live_aligned, QTY, FEE_BPS, EXTRA_BPS, MAX_SKEW, EXCHANGES)
    display(capture_report(live_quotes, live_errors, live_aligned, live_scans, EXCHANGES).round(3))
    print("Retornos nuevos por mercado:", live_returns.count().to_dict())
    display(live_returns.corr(min_periods=20).round(3))
    live_leadlag, _ = leadlag_summary(live_returns, "kraken", ["okx", "bitvavo"], MAX_LAG, .65)
    display(live_leadlag if len(live_leadlag) else "Lead-lag: menos de 20 pares por desplazamiento; no se calcula.")
    display(route_summary(live_scans).round(3))
    display(fee_sensitivity(live_aligned, QTY, [0, 5, 10, 20, 40], EXTRA_BPS, MAX_SKEW, EXCHANGES))
else:
    print("LIVE_E6 = False: no se consulta ningún exchange. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Captura real: muestra insuficiente para lead-lag; ningún arbitraje candidato</b>

**Captura:** 19-11-2026, de 20:23:36 a 20:24:35 UTC (60 rondas, 180 cotizaciones). **Errores: 0**; los tres mercados BTC/EUR respondieron.

| | Kraken | OKX | Bitvavo |
|---|---|---|---|
| Cobertura válida | 100 % | **50,8 %** | 100 % |
| RTT mediano | 381 ms | 599 ms | 399 ms |
| Spread mediano | 0,02 bps | **2,70 bps** | 0,18 bps |
| Retornos nuevos válidos | 34 | 18 | 58 |
| Mid medio frente a Kraken | — | −0,25 € | +1,45 € |

- **OKX pierde la mitad de la cobertura** por el filtro de antigüedad: cada cotización llega entre 0,8 y 1,8 s después de su propio `ts` (mediana 1,2 s). Con la espera hasta el siguiente segundo de la rejilla, el evento tiene unos 2 s y se descarta. Con estos datos no se puede distinguir entre desfase de relojes, latencia o caché de la REST.
- **Kraken** solo tiene 34 retornos nuevos de 59 segundos: con un RTT de unos 380 ms y una ronda por segundo, algunos segundos no reciben cotización nueva y el precio se arrastraría.
- **Correlación y lead-lag:** solo se puede calcular Kraken–Bitvavo (0,29). Con OKX hay menos de 20 pares, y el lead-lag no tiene 20 pares por desplazamiento en ningún segmento. **Muestra insuficiente: no se concluye nada** sobre quién lidera.
- **Arbitraje con 0,01 BTC y 10 bps por lado:** 177 segundos-ruta elegibles y **0 candidatos**. El mejor margen neto fue de −19,3 bps (Kraken → Bitvavo). Diferencia bruta mediana entre los bids y asks cruzados: de −9,5 € a +0,5 €. Solo con comisión 0 aparecen 2 segundos candidatos aislados.
- Los tres precios están a 1–2 € entre sí, frente a unos 22 bps (≈ 125 € por BTC) de costes por ruta: con comisiones realistas, la diferencia normal entre mercados no da para arbitraje.
- **Segunda captura** (20:28 UTC, 25 rondas, 0 errores): OKX tuvo **100 % de cobertura** y la correlación OKX–Bitvavo fue de 0,78 con solo 23 pares. El retraso de OKX y las correlaciones cambian de una captura a otra: con muestras de un minuto no son estables. Arbitraje: de nuevo 0 candidatos con comisiones.

</div>

### E7 · Experimento con WebSocket y latencia de ejecución

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Diseño del experimento</b>

**Pregunta.** ¿Existen diferencias de precio entre Kraken, OKX y Bitvavo en BTC/EUR que superen todos los costes **después** de la latencia de ejecución real?

**Hipótesis (H1).** Cuando el bid de B supera el ask de A en más de un umbral θ (fijado en desarrollo), una compra en A y una venta en B lanzadas en ese instante obtienen un PnL medio positivo después de comisiones, spread, slippage y latencia. **H0:** PnL medio ≤ 0.

**Datos y timestamps (por mensaje):**
- **Fuente:** WebSocket de libro L2 (no solo L1) y de trades de los tres exchanges, con números de secuencia para detectar pérdidas y reconstruir el libro.
- `exchange_at`: hora del evento según el exchange. `received_at`: hora de recepción local, con el reloj sincronizado por NTP/PTP y su desfase registrado.
- `decision_at`: cuándo el sistema detecta la oportunidad; `order_sent_at` y `order_ack_at`: latencia de ida de una orden real o de prueba (`test`/`validate`, sin ejecutar).
- **Latencia de ejecución medida** por exchange (distribución, no un número), con órdenes de prueba o pequeñas órdenes reales en cuenta de pruebas.

**Simulación de ejecución.**
- La orden llega al libro en `decision_at + latencia` (una muestra de la distribución medida). Se ejecuta contra el **libro L2 de ese instante**, recorriendo niveles (VWAP, P02), no contra el L1 de la decisión.
- Comisiones por exchange según el nivel real de la cuenta, mínimos de orden y redondeo.
- **Inventario:** saldos iniciales de EUR y BTC en cada exchange. Una oportunidad solo se ejecuta si hay saldo, y el inventario se actualiza después de cada operación. Se incluye el coste de reequilibrar (retiro y depósito, con sus comisiones y su tiempo) cuando un saldo se agota.
- Riesgo de pierna: si una de las dos órdenes falla o llega tarde, se valora la posición abierta al mid hasta cerrarla.

**Benchmark.** Una estrategia que no hace nada (PnL 0) y otra que opera al azar con la misma frecuencia y tamaño, para separar la señal del ruido y de la exposición.

**Validación.**
- Varias semanas de captura continua. Las primeras para fijar θ, el tamaño y la latencia supuesta; las últimas, nunca vistas, para una única evaluación.
- Métricas: PnL neto total y por operación, tasa de llenado de las dos piernas, distribución de la duración de las oportunidades frente a la latencia, y rentabilidad sobre **todo** el capital inmovilizado (EUR + BTC en los tres exchanges).
- **Rechazo de H1:** si el PnL medio neto en validación es ≤ 0, o si su intervalo de confianza al 95 % (por bloques, por la dependencia temporal) incluye el 0, o si la mayoría de las oportunidades dura menos que la latencia de ejecución medida.

</div>

## 12. Exportación e integración
Si activas `EXPORT`, se guarda la captura original normalizada, los indicadores de rutas y el informe de episodios. El CSV de cotizaciones se puede volver a leer en modo `csv`. Los adaptadores ya están integrados en `src/quantlab/p06/cross_exchange.py`, con sus pruebas en `tests/test_cross_exchange.py`. La captura sintética se etiqueta mediante el nombre del directorio; no mezclarla con datos reales.

In [ ]:
if EXPORT:
    folder=Path(f"p06_output_{DATA_MODE}");folder.mkdir(exist_ok=True)
    quotes.to_csv(folder/"p06_quotes.csv",index=False)
    routes_table.to_csv(folder/"routes.csv")
    leadlag.to_csv(folder/"leadlag.csv",index=False)
    candidate_episodes.to_csv(folder/"episodes.csv",index=False)
    if not errors.empty: errors.to_csv(folder/"capture_errors.csv",index=False)
    print(folder.resolve())
else:
    print("Exportación desactivada. Cambia EXPORT a True si quieres guardar la captura.")

## 13. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P06</b>

**1. ¿Qué relación encontraste entre las correlaciones simultáneas y desplazadas?**
En la simulación, la correlación simultánea es casi nula (0,03 Kraken–OKX, −0,02 Kraken–Bitvavo), pero la desplazada es altísima: ≈ 0,97 (Spearman) con OKX a +2 s y con Bitvavo a +1 s. Los tres mercados siguen la misma trayectoria, solo que con retraso. Una correlación simultánea baja no significa independencia.

**2. ¿La correlación de validación mantiene el signo y magnitud aproximada de desarrollo?**
El signo sí, y el desplazamiento también. La magnitud depende del método: con Pearson cambia mucho (OKX 0,40 → 0,97; Bitvavo 0,98 → 0,29) porque dos desajustes extremos caen uno en cada periodo. Con Spearman se mantiene (≈ 0,97 en los cuatro casos). En la captura real no hubo pares suficientes para calcularla.

**3. ¿Cuántos candidatos sobreviven al tamaño, sincronización y comisiones? ¿Cuántos episodios separados representan?**
En la simulación, con 0,01 BTC, desfase ≤ 750 ms y 10 bps por lado: 16 segundos-ruta en 6 episodios, que en realidad son **2 eventos** (los desajustes inventados). Con 0,1 BTC quedan 4; con 20 bps o más, ninguno. En la captura real: **0 candidatos** con comisiones; solo 2 segundos aislados con comisión 0.

**4. ¿Qué información adicional hace falta para convertir el escáner en una simulación realista?**
Libro L2 por WebSocket con números de secuencia; timestamps del evento en el exchange con relojes sincronizados; latencia de ejecución medida; slippage al recorrer el libro; comisiones reales por nivel de cuenta, mínimos y redondeo; inventario de EUR y BTC en cada exchange y coste de reequilibrar; riesgo de que falle una de las dos piernas; y una validación independiente de varias semanas (ver E7).

**Límite:** no se ha demostrado un alpha; en sintético el retraso y algunos desajustes fueron introducidos a propósito. En datos reales la correlación podría proceder de latencias o factores comunes, y E5 muestra que un simple retraso de recepción crea un "liderazgo" falso. La rentabilidad solo puede evaluarse con ejecución realista, costes completos, inventario y validación independiente.

📄 Resumen en `docs/conclusiones/P06_cross_exchange.md`

</div>

Siguiente práctica: P07 — Derivados financieros, Funding Rate, Open Interest y Basis.